# Análise de Desempenho: Paralelização de Equações Normais

**Disciplina:** INF01008 - Programação Paralela  


**Data de Coleta:** 18 de Abril de 2026  


**Plataforma de Testes:** Cluster GPPD (Hype)  

## 0. Vtune Parsing 

In [33]:
import os
import re
def get_hotspot_function(file_path):
    """
    Extrai o nome da função que mais consumiu tempo (Top Hotspot).
    """
    if not os.path.exists(file_path):
        return "N/A", 0.0
    
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
            
        # Procura o cabeçalho da tabela de hotspots
        for i, line in enumerate(lines):
            if "Function" in line and "Module" in line:
                # A linha seguinte ao cabeçalho é o Top Hotspot
                target_line = lines[i+1]
                parts = [p.strip() for p in target_line.split('\t')]
                
                # Formato esperado: [Hierarquia, Nome da Função, Módulo, CPU Time]
                if len(parts) >= 4:
                    func_name = parts[1]
                    cpu_time = parse_vtune_value(parts[3])
                    return func_name, cpu_time
        return "N/A", 0.0
    except:
        return "Error", 0.0

def get_hpc_metrics_isolated(file_path, metrics_list):
    """
    Extrai métricas do HPC seguindo EXATAMENTE a estrutura:
    Hierarchy Level [TAB] Metric Name [TAB] Metric Value
    """
    results = {m: 0.0 for m in metrics_list}
    
    if not os.path.exists(file_path):
        return results
    
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                # O VTune separa por TABs. Vamos garantir que o split funcione.
                parts = [p.strip() for p in line.split('\t')]
                
                # Se a linha tem pelo menos o nome (parts[1]) e o valor (parts[2])
                if len(parts) >= 3:
                    name_file = parts[1].lower()
                    value_file = parts[2]
                    
                    for m in metrics_list:
                        # Se a métrica que tu queres está na linha
                        if m.lower() == name_file:
                            results[m] = parse_vtune_value(value_file)
                            
        return results
    except Exception as e:
        print(f"⚠️ Erro na leitura física do arquivo: {e}")
        return results

## 1. Baselines

#### 1.1. Data Load


In [ ]:
import pandas as pd
import numpy as np
import os

# Caminhos (ajuste conforme sua pasta local)
PATH_BASE = "./baselines_20260418_1847/app_bench_results.csv"


def load_baseline(path):
    df = pd.read_csv(path)
    
    # 1. Identificar O0 vs O3
    df['opt_level'] = np.where(df.index % 6 < 3, 'O0 (Raw)', 'O3 (Opt)')
    df['threads'] = 1
    
    # --- O FIX ESTÁ AQUI: Incluímos 'dataset' no groupby ---
    group_cols = ['dataset', 'samples', 'features', 'threads', 'opt_level']
    df_mean = df.groupby(group_cols).mean(numeric_only=True).reset_index()
    
    # 2. Criar identificador 'n x m'
    df_mean['problem_size'] = df_mean.apply(lambda x: f"{int(x['samples'])} x {int(x['features'])}", axis=1)
    
    # 3. Ordenar
    df_mean = df_mean.sort_values(['samples', 'features'])
    
    return df_mean


df_baseline = load_baseline(PATH_BASE)

print("✅ Baseline (Serial) carregado e ordenado por complexidade.")
display(df_baseline[['problem_size', 'opt_level', 'fit_time']])

✅ Baseline (Serial) carregado e ordenado por complexidade.


,problem_size,opt_level,fit_time
10,10000 x 15,O0 (Raw),0.005129
11,10000 x 15,O3 (Opt),0.001043
6,100000 x 70,O0 (Raw),0.934061
7,100000 x 70,O3 (Opt),0.090923
8,250000 x 80,O0 (Raw),3.015985
9,250000 x 80,O3 (Opt),0.300399
0,500000 x 100,O0 (Raw),9.101327
1,500000 x 100,O3 (Opt),0.989994
12,515345 x 90,O0 (Raw),7.680374
13,515345 x 90,O3 (Opt),0.835878


### 1.2. Baseline Time

In [17]:
import altair as alt

# Lista para manter a ordem correta dos tamanhos (numérica, não alfabética)
problem_order = df_baseline.sort_values(['samples', 'features'])['problem_size'].unique().tolist()

baseline_viz = alt.Chart(df_baseline).mark_bar().encode(
    x=alt.X('opt_level:N', title=None),
    y=alt.Y('fit_time:Q', title='Tempo (s)'),
    color=alt.Color('opt_level:N', legend=alt.Legend(title="Otimização")),
    column=alt.Column('problem_size:N', sort=problem_order, title='Tamanho do Problema (n x m)'),
    tooltip=['problem_size', 'opt_level', 'fit_time']
).properties(
    width=80,
    title='Desempenho Sequencial: O0 vs O3'
)
baseline_viz

alt.Chart(...)

### 1.3. Spotlight

In [27]:
# Diretórios dos dados
DIR_BASE_VTUNE = "./baselines_20260418_1847/vtune_csvs"

# Configuração das métricas
metrics_cfg = {
    'snap': ['CPI Rate', 'Retiring', 'Back-End Bound'],
    'hpc': ['Memory Bound', 'Vectorization'],
    'hot': ['CPU Time']
}

baseline_analysis_list = []

for _, row in df_baseline.iterrows():
    ds_id = row['dataset'].replace('.csv', '')
    opt_tag = 'raw' if 'O0' in row['opt_level'] else 'opt'
    
    entry = {
        'problem_size': row['problem_size'],
        'samples': row['samples'],
        'features': row['features'],
        'opt_level': row['opt_level'],
        'fit_time': row['fit_time']
    }
    
    # Extração dinâmica para cada tipo de análise
    for a_type, m_list in metrics_cfg.items():
        # Monta o nome pro baseline: tipo_otim_dataset.csv
        fname = f"{a_type}_{opt_tag}_{ds_id}.csv"
        fpath = os.path.join(DIR_BASE_VTUNE, fname)
        
        m_results = get_metrics_from_csv(fpath, m_list)
        entry.update(m_results)
    
    baseline_analysis_list.append(entry)

df_baseline_full = pd.DataFrame(baseline_analysis_list)

print("✅ Métricas do Baseline carregadas com as funções modulares!")
display(df_baseline_full[['problem_size', 'opt_level', 'CPI Rate', 'Vectorization']])

✅ Métricas do Baseline carregadas com as funções modulares!


,problem_size,opt_level,CPI Rate,Vectorization
0,10000 x 15,O0 (Raw),0.0,0.0
1,10000 x 15,O3 (Opt),0.0,0.0
2,100000 x 70,O0 (Raw),0.0,0.0
3,100000 x 70,O3 (Opt),0.0,0.0
4,250000 x 80,O0 (Raw),0.0,0.0
5,250000 x 80,O3 (Opt),0.0,0.0
6,500000 x 100,O0 (Raw),0.0,0.0
7,500000 x 100,O3 (Opt),0.0,0.0
8,515345 x 90,O0 (Raw),0.0,0.0
9,515345 x 90,O3 (Opt),0.0,0.0


### 1.4. Hotspot

In [29]:
# Métricas que vamos tirar do cabeçalho do arquivo Hotspot
hot_metrics = ['CPU Time', 'CPI Rate', 'Instructions Retired']

hotspot_list = []

for _, row in df_baseline.iterrows():
    ds_id = row['dataset'].replace('.csv', '')
    opt_tag = 'raw' if 'O0' in row['opt_level'] else 'opt'
    
    # Caminho do arquivo de hotspot
    fname = f"hot_{opt_tag}_{ds_id}.csv"
    fpath = os.path.join(DIR_BASE_VTUNE, fname)
    
    # 1. Pega métricas gerais
    m_results = get_metrics_from_csv(fpath, hot_metrics)
    
    # 2. Pega o nome da função principal
    top_func, top_func_time = get_hotspot_function(fpath)
    
    entry = {
        'problem_size': row['problem_size'],
        'opt_level': row['opt_level'],
        'top_function': top_func,
        'func_cpu_time': top_func_time,
        **m_results
    }
    hotspot_list.append(entry)

df_hotspots = pd.DataFrame(hotspot_list)

print("✅ Hotspots carregados!")
display(df_hotspots[['problem_size', 'opt_level', 'top_function', 'CPI Rate', 'CPU Time']])

✅ Hotspots carregados!


,problem_size,opt_level,top_function,CPI Rate,CPU Time
0,10000 x 15,O0 (Raw),N/A,0.000000,0.000000
1,10000 x 15,O3 (Opt),N/A,0.000000,0.000000
2,100000 x 70,O0 (Raw),LinearRegression::compute_XtX,0.363077,0.906048
3,100000 x 70,O3 (Opt),LinearRegression::compute_XtX,0.313725,0.065075
4,250000 x 80,O0 (Raw),LinearRegression::compute_XtX,0.365019,2.948411
5,250000 x 80,O3 (Opt),LinearRegression::compute_XtX,0.343373,0.235272
6,500000 x 100,O0 (Raw),LinearRegression::compute_XtX,0.359198,9.015430
7,500000 x 100,O3 (Opt),LinearRegression::compute_XtX,0.413163,0.931077
8,515345 x 90,O0 (Raw),LinearRegression::compute_XtX,0.359343,7.393554
9,515345 x 90,O3 (Opt),LinearRegression::compute_XtX,0.399558,0.745863


In [30]:
import altair as alt

# Preparando os dados para o gráfico (ignorando os zeros do dataset muito pequeno)
df_plot_hot = df_hotspots[df_hotspots['CPU Time'] > 0].copy()

# Gráfico 1: Tempo de CPU (Escala Logarítmica ajuda a ver a diferença brutal)
chart_cpu_time = alt.Chart(df_plot_hot).mark_bar().encode(
    x=alt.X('opt_level:N', title=None),
    y=alt.Y('CPU Time:Q', scale=alt.Scale(type='log'), title='Tempo de CPU (s) - Escala Log'),
    color='opt_level:N',
    column=alt.Column('problem_size:N', sort=problem_order, title='Tamanho do Problema'),
    tooltip=['problem_size', 'opt_level', 'CPU Time']
).properties(width=80, title="Redução de Tempo: O0 vs O3")

# Gráfico 2: CPI Rate (Eficiência de Ciclos)
chart_cpi = alt.Chart(df_plot_hot).mark_bar().encode(
    x=alt.X('opt_level:N', title=None),
    y=alt.Y('CPI Rate:Q', title='CPI Rate (Menor é melhor)'),
    color='opt_level:N',
    column=alt.Column('problem_size:N', sort=problem_order, title='Tamanho do Problema'),
    tooltip=['problem_size', 'opt_level', 'CPI Rate']
).properties(width=80, title="Eficiência do Pipeline (CPI)")

(chart_cpu_time & chart_cpi).resolve_scale(y='independent')

alt.VConcatChart(...)

### 1.5. HPC


In [34]:
# MUDANÇA: Usando apenas o que existe no teu cat
hpc_targets = ['CPI Rate', 'Memory Bound', 'Effective Physical Core Utilization']

hpc_baseline_rows = []

for _, row in df_baseline.iterrows():
    ds_id = row['dataset'].replace('.csv', '')
    opt_tag = 'raw' if 'O0' in row['opt_level'] else 'opt'
    
    # Caminho exato
    fpath = f"{DIR_BASE_VTUNE}/hpc_{opt_tag}_{ds_id}.csv"
    
    # Coleta
    m_data = get_hpc_metrics_isolated(fpath, hpc_targets)
    
    entry = {
        'problem_size': row['problem_size'],
        'opt_level': row['opt_level'],
        **m_data
    }
    hpc_baseline_rows.append(entry)

df_hpc_baseline = pd.DataFrame(hpc_baseline_rows)

print("📊 Tabela de Métricas HPC (Baseline)")
display(df_hpc_baseline)

📊 Tabela de Métricas HPC (Baseline)


,problem_size,opt_level,CPI Rate,Memory Bound,Effective Physical Core Utilization
0,10000 x 15,O0 (Raw),0.000000,0.0,0.0
1,10000 x 15,O3 (Opt),0.000000,0.0,0.0
2,100000 x 70,O0 (Raw),0.378549,0.0,4.3
3,100000 x 70,O3 (Opt),0.311111,0.0,4.1
4,250000 x 80,O0 (Raw),0.371129,2.3,4.9
5,250000 x 80,O3 (Opt),0.351852,21.7,4.0
6,500000 x 100,O0 (Raw),0.360626,1.5,4.9
7,500000 x 100,O3 (Opt),0.419173,15.5,4.6
8,515345 x 90,O0 (Raw),0.367452,2.1,4.9
9,515345 x 90,O3 (Opt),0.408247,14.2,4.9


## 2. Threads

### 2.1 Load data


In [36]:
import pandas as pd
import numpy as np
import os

# Caminho para os resultados das threads
parallel_file = "./results_20260418_1749/app_bench_results.csv"

if os.path.exists(parallel_file):
    # 1. Carregar os dados brutos das execuções com OpenMP
    df_para_raw = pd.read_csv(parallel_file)
    
    # 2. Agrupar pelas características do problema e número de threads
    # Tiramos a média das 3 repetições (HPC, Hot, Snap)
    group_cols = ['dataset', 'samples', 'features', 'threads']
    df_parallel = df_para_raw.groupby(group_cols).mean(numeric_only=True).reset_index()
    
    # 3. Criar a identificação "n x m" para consistência com o baseline
    df_parallel['problem_size'] = df_parallel.apply(
        lambda x: f"{int(x['samples'])} x {int(x['features'])}", axis=1
    )
    
    # 4. Ordenar por tamanho do problema e depois por número de threads
    df_parallel = df_parallel.sort_values(['samples', 'features', 'threads'])
    
    print(f"✅ Dados paralelos carregados! ({len(df_para_raw)} amostras originais)")
    
    # Mostrar as primeiras linhas para conferência
    display(df_parallel[['problem_size', 'threads', 'fit_time', 'total_time']])
else:
    print(f"❌ Erro: Arquivo {parallel_file} não encontrado.")

✅ Dados paralelos carregados! (210 amostras originais)


,problem_size,threads,fit_time,total_time
50,10000 x 15,1,0.001294,0.007146
51,10000 x 15,2,0.001138,0.005464
52,10000 x 15,4,0.002157,0.007164
53,10000 x 15,8,0.002158,0.009569
54,10000 x 15,12,0.002705,0.012772
...,...,...,...,...
25,1000000 x 120,16,0.302695,1.412106
26,1000000 x 120,20,0.260986,1.307942
27,1000000 x 120,24,0.240225,1.292568
28,1000000 x 120,32,0.215415,1.233180


### 2.2 Speedup 

In [39]:
# 1. Isolar as referências do Baseline
df_ref_o0 = df_baseline[df_baseline['opt_level'] == 'O0 (Raw)'][['samples', 'features', 'fit_time']]
df_ref_o0 = df_ref_o0.rename(columns={'fit_time': 't_ref_o0'})

df_ref_o3 = df_baseline[df_baseline['opt_level'] == 'O3 (Opt)'][['samples', 'features', 'fit_time']]
df_ref_o3 = df_ref_o3.rename(columns={'fit_time': 't_ref_o3'})

# 2. Mesclar com os dados paralelos
df_speedup = pd.merge(df_parallel, df_ref_o0, on=['samples', 'features'])
df_speedup = pd.merge(df_speedup, df_ref_o3, on=['samples', 'features'])

# 3. Calcular os dois tipos de Speedup
df_speedup['Speedup (vs O0)'] = df_speedup['t_ref_o0'] / df_speedup['fit_time']
df_speedup['Speedup (vs O3)'] = df_speedup['t_ref_o3'] / df_speedup['fit_time']

# 4. Preparar dados para o Altair (Melt)
# Isso permite criar um seletor no gráfico ou cores diferentes para cada referência
df_speedup_melted = df_speedup.melt(
    id_vars=['problem_size', 'threads'],
    value_vars=['Speedup (vs O0)', 'Speedup (vs O3)'],
    var_name='Referência', value_name='Valor'
)

print("✅ Speedups calculados!")
display(df_speedup[['problem_size', 'threads', 'Speedup (vs O0)', 'Speedup (vs O3)']])

✅ Speedups calculados!


,problem_size,threads,Speedup (vs O0),Speedup (vs O3)
0,10000 x 15,1,3.962658,0.805563
1,10000 x 15,2,4.508350,0.916496
2,10000 x 15,4,2.378207,0.483462
3,10000 x 15,8,2.376738,0.483163
4,10000 x 15,12,1.895885,0.385412
...,...,...,...,...
65,1000000 x 120,16,85.742307,9.116844
66,1000000 x 120,20,99.445184,10.573849
67,1000000 x 120,24,108.039262,11.487644
68,1000000 x 120,32,120.482639,12.810728


In [42]:
import pandas as pd
import numpy as np
import altair as alt
import os

# ==========================================
# 1. CONFIGURAÇÕES E CARGA DE DADOS
# ==========================================
PATH_BASE = "./baselines_20260418_1847/app_bench_results.csv"
PATH_PARA = "./results_20260418_1749/app_bench_results.csv"

def load_and_clean_data(path, is_baseline=False):
    df = pd.read_csv(path)
    
    # Identificação de O0 vs O3 para o baseline (blocos de 6 linhas)
    if is_baseline:
        df['opt_level'] = np.where(df.index % 6 < 3, 'O0 (Raw)', 'O3 (Opt)')
        df['threads'] = 1
        group_cols = ['dataset', 'samples', 'features', 'threads', 'opt_level']
    else:
        group_cols = ['dataset', 'samples', 'features', 'threads']
    
    # Agrupamento e média das repetições
    df_mean = df.groupby(group_cols).mean(numeric_only=True).reset_index()
    
    # Identificador amigável n x m
    df_mean['problem_size'] = df_mean.apply(lambda x: f"{int(x['samples'])} x {int(x['features'])}", axis=1)
    return df_mean

df_baseline = load_and_clean_data(PATH_BASE, is_baseline=True)
df_parallel = load_and_clean_data(PATH_PARA, is_baseline=False)

# ==========================================
# 2. CÁLCULO DE SPEEDUP
# ==========================================
# Isolar as referências
df_ref_o0 = df_baseline[df_baseline['opt_level'] == 'O0 (Raw)'][['samples', 'features', 'fit_time']].rename(columns={'fit_time': 't_ref_o0'})
df_ref_o3 = df_baseline[df_baseline['opt_level'] == 'O3 (Opt)'][['samples', 'features', 'fit_time']].rename(columns={'fit_time': 't_ref_o3'})

# Mesclar com os dados paralelos
df_analysis = pd.merge(df_parallel, df_ref_o0, on=['samples', 'features'])
df_analysis = pd.merge(df_analysis, df_ref_o3, on=['samples', 'features'])

# Calcular as métricas de Speedup
df_analysis['Speedup (vs O0)'] = df_analysis['t_ref_o0'] / df_analysis['fit_time']
df_analysis['Speedup (vs O3)'] = df_analysis['t_ref_o3'] / df_analysis['fit_time']

# Preparar para o gráfico (Melt)
df_plot = df_analysis.melt(
    id_vars=['problem_size', 'threads', 'samples', 'features'],
    value_vars=['Speedup (vs O0)', 'Speedup (vs O3)'],
    var_name='Referência', value_name='Speedup'
)

# ==========================================
# 3. FILTRAGEM E VISUALIZAÇÃO (ALTAIR)
# ==========================================
# 1. Definir a ordem correta dos datasets (numérica)
problem_order = df_analysis.sort_values(['samples', 'features'])['problem_size'].unique().tolist()

# 2. Filtrar datasets muito pequenos que geram ruído (opcional)
# Aqui removemos o 'tiny_test' e o 'medium_bench' se necessário
df_filtered = df_plot[~df_plot['problem_size'].str.contains('10000 x 15|100000 x 70')].copy()

# 3. Criar a Linha de Speedup Ideal (para o gráfico vs O3)
ideal_df = pd.DataFrame({'threads': [1, 40], 'Speedup': [1, 40], 'Referência': ['Speedup (vs O3)', 'Speedup (vs O3)']})
ideal_line = alt.Chart(ideal_df).mark_line(strokeDash=[5,5], color='gray', opacity=0.5).encode(
    x='threads:Q', y='Speedup:Q'
)

# 4. Gráfico Facetado com Escala Logarítmica
speedup_chart = alt.Chart(df_filtered).mark_line(point=True).encode(
    x=alt.X('threads:Q', title='Número de Threads'),
    y=alt.Y('Speedup:Q', 
           scale=alt.Scale(type='log'), 
           title='Speedup (Escala Log)'),
    color=alt.Color('problem_size:N', sort=problem_order, title='Tamanho do Problema'),
    tooltip=['problem_size', 'threads', 'Speedup']
).properties(
    width=350,
    height=400,
    title='Análise de Escalabilidade Paralela'
).facet(
    column=alt.Column('Referência:N', title='Baseline de Referência')
).resolve_scale(
    y='independent' # Permite que cada gráfico tenha seu próprio range de valores
)

speedup_chart.interactive()


alt.FacetChart(...)

### 2.3. Hotspot

In [46]:
# Diretório dos resultados das threads
DIR_PARA_VTUNE = "./results_20260418_1749/vtune_csvs"

# Métricas que vamos extrair do arquivo de Hotspot paralelo
hot_para_metrics = ['CPU Time', 'Elapsed Time', 'Instructions Retired', 'CPI Rate']

hot_para_list = []

# Varremos o df_parallel (que já tem o mapeamento de datasets e threads)
for _, row in df_parallel.iterrows():
    ds_id = row['dataset'].replace('.csv', '')
    t = int(row['threads'])
    
    # O padrão de nome das threads: hot_dataset_tX.csv
    fname = f"hot_{ds_id}_t{t}.csv"
    fpath = os.path.join(DIR_PARA_VTUNE, fname)
    
    # 1. Pegamos as métricas de cabeçalho
    m_data = get_hpc_metrics_isolated(fpath, hot_para_metrics)
    
    # 2. Pegamos a função principal (para confirmar se o trabalho continua nela)
    top_func, top_func_time = get_hotspot_function(fpath)
    
    entry = {
        'problem_size': row['problem_size'],
        'threads': t,
        'top_function': top_func,
        'func_cpu_time': top_func_time,
        **m_data
    }
    hot_para_list.append(entry)

df_hot_para = pd.DataFrame(hot_para_list)

# Criar métrica de "Utilização Real"
# Se for 1.0, todas as threads trabalharam o tempo todo. 
# Se for 0.5, as threads ficaram 50% do tempo esperando.
df_hot_para['Core Utilization Factor'] = df_hot_para['CPU Time'] / (df_hot_para['Elapsed Time'] * df_hot_para['threads'])

print("✅ Hotspots das Threads carregados!")
display(df_hot_para[['problem_size', 'threads', 'CPU Time', 'Elapsed Time', 'CPI Rate']].head(10))

✅ Hotspots das Threads carregados!


,problem_size,threads,CPU Time,Elapsed Time,CPI Rate
0,500000 x 100,1,0.896037,2.582716,0.433400
1,500000 x 100,2,0.891031,1.518373,0.469565
2,500000 x 100,4,0.946095,1.010040,0.494624
3,500000 x 100,8,1.596848,0.762269,0.708911
4,500000 x 100,12,1.301506,0.649600,0.631692
5,500000 x 100,16,1.752027,0.634812,0.789264
6,500000 x 100,20,2.562966,0.670832,1.147929
7,500000 x 100,24,0.220255,0.642629,1.020408
8,500000 x 100,32,0.815944,0.593492,0.859223
9,500000 x 100,40,3.554111,0.606876,1.620352


In [47]:
import altair as alt

util_chart = alt.Chart(df_hot_para).mark_line(point=True).encode(
    x=alt.X('threads:Q', title='Número de Threads'),
    y=alt.Y('Core Utilization Factor:Q', title='Fator de Utilização (1.0 = Ideal)'),
    color=alt.Color('problem_size:N', sort=problem_order, title='Tamanho do Problema'),
    tooltip=['problem_size', 'threads', 'Core Utilization Factor', 'CPU Time']
).properties(
    width=600, height=400,
    title='Eficiência do Escalonamento OpenMP'
).interactive()

util_chart

alt.Chart(...)

### 2.4. HPC

In [44]:
# Pasta onde estão os resultados das threads
DIR_PARA_VTUNE = "./results_20260418_1749/vtune_csvs"

# Métricas que vamos extrair (as que o HPC fornece com precisão)
hpc_para_targets = ['CPI Rate', 'Memory Bound', 'Cache Bound']

para_hpc_list = []

# Varremos o df_parallel (que tem o mapeamento de datasets e threads executadas)
for _, row in df_parallel.iterrows():
    ds_id = row['dataset'].replace('.csv', '')
    t = int(row['threads'])
    
    # Monta o caminho seguindo o padrão: hpc_dataset_tX.csv
    fname = f"hpc_{ds_id}_t{t}.csv"
    fpath = os.path.join(DIR_PARA_VTUNE, fname)
    
    # Coleta usando o parser de baixo nível (independente de Pandas)
    m_data = get_hpc_metrics_isolated(fpath, hpc_para_targets)
    
    entry = {
        'problem_size': row['problem_size'],
        'threads': t,
        'samples': row['samples'],
        'features': row['features'],
        **m_data
    }
    para_hpc_list.append(entry)

df_hpc_para = pd.DataFrame(para_hpc_list)

print("✅ Métricas HPC das Threads carregadas com sucesso!")
display(df_hpc_para)

✅ Métricas HPC das Threads carregadas com sucesso!


,problem_size,threads,samples,features,CPI Rate,Memory Bound,Cache Bound
0,500000 x 100,1,500000,100,0.458252,21.6,10.3
1,500000 x 100,2,500000,100,0.518664,31.1,20.8
2,500000 x 100,4,500000,100,0.500000,30.0,19.6
3,500000 x 100,8,500000,100,0.531373,36.1,20.2
4,500000 x 100,12,500000,100,0.584980,28.0,35.0
...,...,...,...,...,...,...,...
65,515345 x 90,16,515345,90,0.668657,42.7,29.9
66,515345 x 90,20,515345,90,0.815242,16.0,32.8
67,515345 x 90,24,515345,90,0.988345,74.4,34.5
68,515345 x 90,32,515345,90,1.154195,62.9,38.3


In [45]:
import altair as alt

# Ordenação para os gráficos
order = df_hpc_para.sort_values(['samples', 'features'])['problem_size'].unique().tolist()

# Gráfico 1: Evolução do Gargalo de Memória
mem_chart = alt.Chart(df_hpc_para).mark_line(point=True).encode(
    x=alt.X('threads:Q', title='Número de Threads (OpenMP)'),
    y=alt.Y('Memory Bound:Q', title='Memory Bound (%)'),
    color=alt.Color('problem_size:N', sort=order, title='Tamanho do Problema'),
    tooltip=['problem_size', 'threads', 'Memory Bound']
).properties(
    width=500, height=350,
    title='Saturação da Memória por Thread'
)

# Gráfico 2: Evolução da Eficiência de Ciclo (CPI)
cpi_para_chart = alt.Chart(df_hpc_para).mark_line(point=True).encode(
    x=alt.X('threads:Q', title='Número de Threads (OpenMP)'),
    y=alt.Y('CPI Rate:Q', title='CPI Rate (Ciclos por Instrução)'),
    color=alt.Color('problem_size:N', sort=order, title='Tamanho do Problema'),
    tooltip=['problem_size', 'threads', 'CPI Rate']
).properties(
    width=500, height=350,
    title='Eficiência de Execução (CPI) por Thread'
)

(mem_chart | cpi_para_chart).resolve_scale(y='independent')

alt.HConcatChart(...)